In [1]:
!pip install pytorch-fid torchmetrics gradio -q

In [2]:
!git clone -b Frontend https://github.com/youlkar/damage-diffusion.git

fatal: destination path 'damage-diffusion' already exists and is not an empty directory.


In [3]:
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import io, os

# Authenticate with your Google account (same one that has access to the file)
auth.authenticate_user()
service = build('drive', 'v3')

FILE_ID = '1VUfqiFdi8HJfcJY6J1931Je1QBw_t72O'
OUTPUT  = '/content/final_model.pt'
print("Starting download via Drive API...", flush=True)
request    = service.files().get_media(fileId=FILE_ID)
fh         = io.FileIO(OUTPUT, 'wb')
downloader = MediaIoBaseDownload(fh, request, chunksize=100 * 1024 * 1024)  # 100MB chunks

done = False
while not done:
    status, done = downloader.next_chunk()
    pct = int(status.progress() * 100)
    print(f'\rDownloading... {pct}%', end='', flush=True)

fh.close()
print(f'\nDone! Size: {os.path.getsize(OUTPUT) / 1e9:.2f} GB', flush=True)
CKPT_PATH = OUTPUT
print(f'Use this path: {CKPT_PATH}', flush=True)

Starting download via Drive API...


Downloading... 100%
Done! Size: 1.70 GB
Use this path: /content/final_model.pt


In [4]:
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import io, os

auth.authenticate_user()
service = build('drive', 'v3')

FILE_ID = '1v2uGKKTtm9LHdb1uRkfWqPxJ3oG2i5xy'
OUTPUT = '/content/metrics.pt'

print("Starting download via Drive API...", flush=True)
request = service.files().get_media(fileId=FILE_ID)
fh = io.FileIO(OUTPUT, 'wb')
downloader = MediaIoBaseDownload(fh, request, chunksize=100 * 1024 * 1024)
done = False
while not done:
    status, done = downloader.next_chunk()
    pct = int(status.progress() * 100)
    print(f'\rDownloading... {pct}%', end='', flush=True)
fh.close()
print(f'\nDone! Size: {os.path.getsize(OUTPUT)/1e6:.2f} MB', flush=True)
print(f'Use this path: {OUTPUT}', flush=True)

Starting download via Drive API...


Downloading... 100%
Done! Size: 0.00 MB
Use this path: /content/metrics.pt


In [5]:
import torch

metrics = torch.load('/content/metrics.pt')
print("Keys:", list(metrics.keys()))
for k, v in metrics.items():
    if isinstance(v, list):
        print(f"{k}: {len(v)} values, first 5: {v[:5]}")
    else:
        print(f"{k}: {v}")

Keys: ['train_loss', 'val_loss', 'fid_score', 'kid_score', 'mask_sensitivity_score', 'learning_rate', 'epoch']
train_loss: 100 values, first 5: [0.06754126211866564, 0.02202778965841199, 0.02031213676199577, 0.017912644762978998, 0.01609308089042992]
val_loss: 100 values, first 5: [0.034065761230885984, 0.028940191306173802, 0.02213093126192689, 0.019200839474797247, 0.022374577820301056]
fid_score: 10 values, first 5: [184.26223754882812, 200.0268096923828, 142.72586059570312, 133.09805297851562, 167.7210235595703]
kid_score: 10 values, first 5: [0.13580086827278137, 0.14245033264160156, 0.08864492177963257, 0.09297499060630798, 0.12517580389976501]
mask_sensitivity_score: 10 values, first 5: [141.3406524658203, 141.23789978027344, 141.66444396972656, 141.21377563476562, 141.54226684570312]
learning_rate: 100 values, first 5: [9.997557473810373e-05, 9.990232305719946e-05, 9.978031724785248e-05, 9.960967771506668e-05, 9.939057285945934e-05]
epoch: 100 values, first 5: [0, 1, 2, 3, 4]


In [6]:
import sys, os
# Removed sys.path.insert and os.chdir from this cell to avoid conflicts.
os.system('git checkout Frontend')

32768

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
import sys, os, importlib
sys.path.insert(0, '/content/damage-diffusion/backend')
spec = importlib.util.find_spec('inference')
print('find_spec:', spec)
if spec:
    print('origin:', spec.origin)

find_spec: ModuleSpec(name='inference', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7bc2189497f0>, origin='/content/damage-diffusion/backend/inference.py')
origin: /content/damage-diffusion/backend/inference.py


In [10]:
import gradio as gr
import torch
import numpy as np
from PIL import Image
import torchvision.transforms as transforms
import sys, os

sys.path.insert(0, '/content/damage-diffusion/backend')
from inference import load_model, generate_from_mask
from utils.metrics import compute_iou, compute_pixel_accuracy
from utils.visualization import denormalize

model = None
device = None

def load(checkpoint_path, device_choice):
    global model, device
    if device_choice == "cuda" and not torch.cuda.is_available():
        device_choice = "cpu"
    device = device_choice
    if not os.path.exists(checkpoint_path):
        return f"ERROR: Checkpoint not found at {checkpoint_path}"
    try:
        model, _ = load_model(checkpoint_path, device=device)
        return f"Model loaded from {checkpoint_path} on {device}"
    except Exception as e:
        return f"ERROR loading model: {str(e)}"

def preprocess_mask(mask_img, image_size=128, invert=False):
    if isinstance(mask_img, dict):
        if mask_img.get('composite') is not None:
            mask_img = mask_img['composite']
        elif mask_img.get('image') is not None:
            mask_img = mask_img['image']
        else:
            mask_img = list(mask_img.values())[0]
    pil = Image.fromarray(mask_img.astype(np.uint8)).convert('L')
    pil = pil.resize((image_size, image_size))
    t = transforms.ToTensor()(pil)
    t = (t < 0.5).float() if invert else (t > 0.5).float()
    return t.unsqueeze(0)

def has_drawing(draw):
    if draw is None:
        return False
    if isinstance(draw, dict):
        arr = draw.get('composite')
        if arr is None:
            arr = draw.get('image')
    else:
        arr = draw
    if arr is None:
        return False
    return bool(np.array(arr).min() < 50)

def tensor_to_pil(t):
    t = t.squeeze(0).cpu()
    t = denormalize(t).clamp(0, 1)
    return transforms.ToPILImage()(t)

def generate(upload, draw, num_steps, num_samples):
    if model is None:
        return None, 'Load a model checkpoint first.'
    if has_drawing(draw):
        mask_np    = draw
        use_invert = True   # sketchpad: black strokes on white bg
    elif upload is not None:
        mask_np    = upload
        use_invert = False  # dataset masks: white cracks on black bg
    else:
        return None, 'Please upload or draw a mask first.'
    mask_tensor = preprocess_mask(mask_np, invert=use_invert)
    generated   = generate_from_mask(
        model, mask_tensor,
        num_samples=int(num_samples),
        num_inference_steps=int(num_steps),
        device=device,
    )
    return tensor_to_pil(generated[0]), 'Done!'

def evaluate(gen_img, gt_mask):
    if gen_img is None or gt_mask is None:
        return "Please provide both images."
    gen_t      = transforms.ToTensor()(Image.fromarray(gen_img)).unsqueeze(0)
    mask_t     = preprocess_mask(gt_mask)
    gen_gray   = gen_t.mean(dim=1, keepdim=True)
    gen_binary = (gen_gray > 0.5).float()
    iou = compute_iou(gen_binary, mask_t)
    acc = compute_pixel_accuracy(gen_binary, mask_t)
    return f"IoU: {iou:.4f}\nPixel Accuracy: {acc:.4f}"

available_models = [f for f in ['/content/final_model.pt', '/content/best_model.pt'] if os.path.exists(f)]

with gr.Blocks(title="DamageDiffusion") as demo:
    gr.Markdown("# DamageDiffusion — Crack Image Generator")
    with gr.Row():
        ckpt_input   = gr.Dropdown(choices=available_models, value=available_models[0], label="Checkpoint", allow_custom_value=True) if available_models else gr.Textbox(label="Checkpoint path", value="/content/final_model.pt")
        device_drop  = gr.Dropdown(choices=["cuda", "cpu"], value="cuda", label="Device")
        load_btn     = gr.Button("Load Model")
        load_status  = gr.Textbox(label="Status", interactive=False)
        load_btn.click(load, inputs=[ckpt_input, device_drop], outputs=load_status)
    with gr.Tabs():
        with gr.Tab("Generate"):
            gr.Markdown("**Option 1:** Upload mask  —  **Option 2:** Draw below. Hit Generate.")
            with gr.Row():
                with gr.Column():
                    mask_upload = gr.Image(label="Option 1 — Upload mask", type="numpy", sources=["upload"])
                    gr.Markdown("— OR —")
                    mask_draw   = gr.Sketchpad(label="Option 2 — Draw mask", type="numpy", image_mode="RGB", canvas_size=(512, 512))
                    num_steps   = gr.Slider(10, 1000, value=50, step=10, label="Inference steps")
                    num_samples = gr.Slider(1, 4, value=1, step=1, label="Num samples")
                    gen_btn     = gr.Button("Generate", variant="primary")
                with gr.Column():
                    gen_output  = gr.Image(label="Generated image")
                    gen_status  = gr.Textbox(label="Status", interactive=False)
            gen_btn.click(generate, inputs=[mask_upload, mask_draw, num_steps, num_samples], outputs=[gen_output, gen_status])
        with gr.Tab("Evaluate"):
            gr.Markdown("Upload a generated image and a ground truth mask.")
            with gr.Row():
                eval_gen    = gr.Image(label="Generated image", type="numpy", sources=["upload"])
                eval_mask   = gr.Image(label="Ground truth mask", type="numpy", sources=["upload"])
                eval_btn    = gr.Button("Compute metrics", variant="primary")
                eval_output = gr.Textbox(label="Metrics", interactive=False, lines=6)
                eval_btn.click(evaluate, inputs=[eval_gen, eval_mask], outputs=eval_output)

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://72695926a70970bfaf.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Loaded training config:
Model channels: (128, 256, 512, 512)
Timesteps: 1000
Image size: 128
Loaded model from /content/final_model.pt
Trained for 99 epochs
Generating 1 samples with 50 steps...
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://72695926a70970bfaf.gradio.live
